# Сегментация воды и оценка затопления — Sen1Floods11

**Курсовая:** «Терең оқыту негізінде спутниктік суреттерден су басқан аумақтарды анықтау және сегментациялау».

U-Net + ResNet-34, VV/VH, BCE + Dice. Метрики воды отделены от оценки затопления.
Исходные данные: [Sen1Floods11](https://github.com/cloudtostreet/Sen1Floods11), Bonafilia et al., CVPR Workshops 2020.

Выберите **Среда выполнения → Сменить среду выполнения → T4 GPU** (если доступен).
Запускайте ячейки сверху вниз. Подключение Drive требует вашего разрешения: код сохраняет результаты в `MyDrive/FloodSegmentation`.
Google может прервать сессию; `last.pt` позволяет продолжить. После перезапуска повторите настройку, подключение Drive и загрузку данных, затем ячейку обучения.
Данные хранятся в `/content`, результаты — на Drive. Публичная публикация не требуется.


## 1. Код проекта и зависимости
PyTorch из Colab сохраняется. Устанавливаются недостающие библиотеки.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
WORK = Path('/content/flood_project')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
if str(WORK) not in sys.path: sys.path.insert(0, str(WORK))
SOURCES = {'flood/core.py': "import json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\ndef write_json(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    def clean(x):\n        if isinstance(x, dict): return {str(k):clean(v) for k,v in x.items()}\n        if isinstance(x, (tuple,list)): return [clean(v) for v in x]\n        if isinstance(x, np.generic): return clean(x.item())\n        if isinstance(x, float) and not np.isfinite(x): return None\n        return x\n    path.write_text(json.dumps(clean(value), indent=2, ensure_ascii=False), encoding='utf-8')\n\ndef counts(pred, target, valid):\n    p=np.asarray(pred,dtype=bool); y=np.asarray(target,dtype=bool); v=np.asarray(valid,dtype=bool)\n    return np.array([(p&y&v).sum(),(p&~y&v).sum(),(~p&y&v).sum(),(~p&~y&v).sum()],dtype=np.int64)\n\ndef metrics(c):\n    tp,fp,fn,tn=map(int,c)\n    def div(a,b): return a/b if b else float('nan')\n    return dict(iou=div(tp,tp+fp+fn), f1=div(2*tp,2*tp+fp+fn), precision=div(tp,tp+fp), recall=div(tp,tp+fn), tp=tp,fp=fp,fn=fn,tn=tn)\n\ndef masked_loss(logits,target,valid):\n    if not valid.any(): return None\n    # Index BEFORE BCE so NaN/unknown values outside valid cannot propagate.\n    z=logits[valid].float(); y=target[valid].float(); p=z.sigmoid()\n    return F.binary_cross_entropy_with_logits(z,y)+1-(2*(p*y).sum()+1e-6)/(p.sum()+y.sum()+1e-6)\n\ndef assemble(patches, size=512, patch=256):\n    if size%patch or len(patches)!=(size//patch)**2: raise ValueError('Incomplete tiling')\n    out=np.zeros((size,size),dtype=np.asarray(patches[0]).dtype)\n    for k,a in enumerate(patches):\n        r,c=divmod(k,size//patch); out[r*patch:(r+1)*patch,c*patch:(c+1)*patch]=a\n    return out\n\ndef validate_splits(rows):\n    seen=set()\n    for r in rows:\n        if r['chip_id'] in seen: raise ValueError('Duplicate/leaking chip: '+r['chip_id'])\n        seen.add(r['chip_id'])\n        if (r['event']=='Bolivia') != (r['split']=='bolivia'): raise ValueError('Bolivia holdout leakage')\n", 'flood/data.py': "import csv, hashlib, io, json, time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport requests\nimport rasterio\nfrom rasterio.windows import Window\nimport torch\nfrom torch.utils.data import Dataset\nfrom .core import validate_splits, write_json\n\nBASE='https://storage.googleapis.com/sen1floods11/v1.1/'\nHAND='data/flood_events/HandLabeled/'\n\ndef download(url,path):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    if path.exists() and path.stat().st_size:\n        try:\n            if path.suffix=='.tif':\n                with rasterio.open(path) as ds: ds.read()\n            return\n        except rasterio.errors.RasterioError: pass\n    temp=path.with_suffix(path.suffix+'.part')\n    for attempt in range(3):\n        try:\n            with requests.get(url,stream=True,timeout=(20,120)) as r:\n                r.raise_for_status()\n                with temp.open('wb') as f:\n                    for chunk in r.iter_content(1024*1024): f.write(chunk)\n            temp.replace(path); return\n        except requests.RequestException:\n            if attempt==2: raise\n            time.sleep(attempt+1)\n\ndef prepare(root, include_jrc=True, limit=None):\n    root=Path(root); rows=[]; jobs=[]\n    for split,remote in [('train','train'),('valid','valid'),('test','test'),('bolivia','bolivia')]:\n        p=root/'splits'/f'flood_{remote}_data.csv'\n        download(BASE+'splits/flood_handlabeled/'+p.name,p)\n        pairs=[r for r in csv.reader(io.StringIO(p.read_text())) if r]\n        if limit is not None: pairs=pairs[:limit]\n        for image,label in pairs:\n            chip=image.removesuffix('_S1Hand.tif')\n            row=dict(chip_id=chip,event=chip.split('_')[0],split=split)\n            for key,folder,name in [('image','S1Hand',image),('label','LabelHand',label)]+([('jrc','JRCWaterHand',chip+'_JRCWaterHand.tif')] if include_jrc else []):\n                dest=root/folder/name; row[key]=str(dest.resolve()); jobs.append((BASE+HAND+folder+'/'+name,dest))\n            rows.append(row)\n    validate_splits(rows)\n    with ThreadPoolExecutor(max_workers=6) as pool:\n        for i,_ in enumerate(pool.map(lambda a:download(*a),jobs),1):\n            if i%100==0 or i==len(jobs): print(f'Downloaded/verified {i}/{len(jobs)}',flush=True)\n    df=pd.DataFrame(rows); df.to_csv(root/'manifest.csv',index=False)\n    return df\n\ndef read_image(row, window=None):\n    with rasterio.open(row['image']) as ds:\n        x=ds.read(window=window).astype('float32'); iv=(ds.read_masks(window=window)>0).all(0)\n    return x,iv&np.isfinite(x).all(0)\n\ndef read_pair(row, window=None):\n    x,iv=read_image(row,window)\n    with rasterio.open(row['label']) as ds:\n        y=ds.read(1,window=window); lv=ds.read_masks(1,window=window)>0\n    valid=iv&lv&np.isfinite(x).all(0)&np.isin(y,[0,1])\n    return x,(y==1).astype('float32'),valid\n\ndef audit(df,out):\n    out=Path(out); out.mkdir(parents=True,exist_ok=True); records=[]; bounds=[]\n    sums=np.zeros(2); squares=np.zeros(2); n=0\n    for row in df.to_dict('records'):\n        with rasterio.open(row['image']) as a, rasterio.open(row['label']) as b:\n            if a.count!=2 or a.shape!=(512,512) or b.count!=1 or a.shape!=b.shape or a.crs!=b.crs or not a.transform.almost_equals(b.transform):\n                raise ValueError('Raster geometry mismatch: '+row['chip_id'])\n            values=np.unique(b.read(1))\n            if not set(values).issubset({-1,0,1}): raise ValueError(f'Unexpected labels: {values}')\n            bounds.append((row['chip_id'],row['split'],str(a.crs),a.bounds))\n        x,y,v=read_pair(row)\n        rec={k:row[k] for k in ['chip_id','event','split']}\n        rec.update(valid_pixels=int(v.sum()),water_pixels=int(y[v].sum()),invalid_fraction=float(1-v.mean()))\n        for key in ['image','label']:\n            p=Path(row[key]); rec[key+'_bytes']=p.stat().st_size; rec[key+'_sha256']=hashlib.sha256(p.read_bytes()).hexdigest()\n        records.append(rec)\n        if row['split']=='train':\n            _,image_valid=read_image(row)\n            z=x[:,image_valid].astype('float64'); sums+=z.sum(1); squares+=(z*z).sum(1); n+=z.shape[1]\n    overlaps=[]\n    for i,(aid,asp,crs,a) in enumerate(bounds):\n        for bid,bsp,bcrs,b in bounds[i+1:]:\n            if asp!=bsp and crs==bcrs and min(a.right,b.right)-max(a.left,b.left)>1e-8 and min(a.top,b.top)-max(a.bottom,b.bottom)>1e-8:\n                overlaps.append([aid,bid])\n    write_json(out/'spatial_overlaps.json',overlaps)\n    if overlaps: raise ValueError('Spatial overlap between splits; inspect report before training')\n    if not n: raise ValueError('No valid training pixels')\n    mean=sums/n; std=np.sqrt(np.maximum(squares/n-mean**2,1e-12))\n    norm={'mean':mean.tolist(),'std':std.tolist(),'train_valid_pixels':n,'units':'dB'}\n    write_json(out/'normalization.json',norm)\n    stats=pd.DataFrame(records); stats.to_csv(out/'audit.csv',index=False)\n    stats.groupby(['split','event']).agg(tiles=('chip_id','count'),valid_pixels=('valid_pixels','sum'),water_pixels=('water_pixels','sum')).to_csv(out/'split_summary.csv')\n    return norm,stats\n\ndef normalize(x, valid, norm):\n    z=(x-np.asarray(norm['mean'],dtype='float32')[:,None,None])/np.asarray(norm['std'],dtype='float32')[:,None,None]\n    z[:,~valid]=0\n    return z\n\nclass FloodDataset(Dataset):\n    def __init__(self,df,split,norm,augment=False):\n        self.rows=df[df.split==split].to_dict('records'); self.norm=norm; self.augment=augment; self.items=[]; self.skipped=0\n        for i,r in enumerate(self.rows):\n            _,_,v=read_pair(r)\n            for top in [0,256]:\n                for left in [0,256]:\n                    if v[top:top+256,left:left+256].any(): self.items.append((i,top,left))\n                    else: self.skipped+=1\n    def __len__(self): return len(self.items)\n    def __getitem__(self,k):\n        i,top,left=self.items[k]; row=self.rows[i]\n        window=Window(left,top,256,256)\n        x,y,v=read_pair(row,window); _,image_valid=read_image(row,window); x=normalize(x,image_valid,self.norm)\n        y=y[None]; v=v[None]\n        if self.augment:\n            turns=int(torch.randint(4,()).item()); flip=bool(torch.rand(())<.5)\n            arrays=[np.rot90(a,turns,axes=(-2,-1)) for a in [x,y,v]]\n            x,y,v=[np.flip(a,-1) if flip else a for a in arrays]\n        return {'image':torch.from_numpy(x.copy()),'target':torch.from_numpy(y.copy()),'valid':torch.from_numpy(v.copy()),'chip_id':row['chip_id'],'top':top,'left':left}\n\ndef eda(df,out):\n    import matplotlib.pyplot as plt\n    out=Path(out); selected=df[df.split=='train'].groupby('event').head(1).head(8)\n    for row in selected.to_dict('records'):\n        x,y,v=read_pair(row); fig,axes=plt.subplots(1,4,figsize=(14,4))\n        for ax,a,title,cmap in zip(axes,[x[0],x[1],np.where(v,y,np.nan),v],['VV (dB)','VH (dB)','Water label','Valid pixels'],['gray','gray','Blues','gray']):\n            ax.imshow(a,cmap=cmap); ax.set_title(title); ax.axis('off')\n        fig.suptitle(row['chip_id']); fig.tight_layout(); fig.savefig(out/(row['chip_id']+'_data.png'),dpi=180); plt.close(fig)\n", 'flood/evaluate.py': "from pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nimport rasterio\nfrom rasterio.warp import reproject,Resampling\nfrom .data import read_pair,read_image,normalize\nfrom .models import build_model\nfrom .core import counts,metrics,assemble,write_json\n\ndef load_model(checkpoint,device=None):\n    device=torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))\n    state=torch.load(checkpoint,map_location='cpu',weights_only=False)\n    model=build_model(state['config']['model'],False).to(device)\n    model.load_state_dict(state['model']); model.eval()\n    return model,state['normalization'],device\n\n@torch.inference_mode()\ndef predict(model,x,valid,norm,device):\n    z=normalize(x,valid,norm); patches=[]\n    for top in [0,256]:\n        for left in [0,256]:\n            a=torch.from_numpy(z[:,top:top+256,left:left+256].copy())[None].to(device)\n            patches.append(model(a).sigmoid()[0,0].cpu().numpy())\n    return assemble(patches)\n\ndef otsu(x,valid):\n    from skimage.filters import threshold_otsu\n    values=x[1][valid]\n    if not len(values) or np.ptp(values)==0: return np.zeros_like(valid)\n    return x[1]<threshold_otsu(values)\n\ndef choose_threshold(df,model,norm,device,out):\n    thresholds=np.round(np.arange(.1,.901,.05),2); totals=np.zeros((len(thresholds),4),dtype=np.int64)\n    for row in df[df.split=='valid'].to_dict('records'):\n        x,y,v=read_pair(row); _,image_valid=read_image(row); prob=predict(model,x,image_valid,norm,device)\n        for i,t in enumerate(thresholds): totals[i]+=counts(prob>=t,y,v)\n    scores=[metrics(c)['iou'] for c in totals]\n    order=sorted(range(len(thresholds)),key=lambda i:(-scores[i],abs(thresholds[i]-.5),thresholds[i]))\n    threshold=float(thresholds[order[0]])\n    write_json(Path(out)/'threshold.json',{'threshold':threshold,'source':'valid','curve':[{'threshold':float(t),'iou':s} for t,s in zip(thresholds,scores)]})\n    return threshold\n\ndef export_raster(source,path,array,valid,probability=False):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    nodata=-9999 if probability else 255; dtype='float32' if probability else 'uint8'\n    with rasterio.open(source) as ds: profile=ds.profile.copy()\n    profile.update(count=1,dtype=dtype,nodata=nodata,compress='deflate')\n    with rasterio.open(path,'w',**profile) as dst: dst.write(np.where(valid,array,nodata).astype(dtype),1)\n\ndef aligned_jrc(row):\n    with rasterio.open(row['image']) as dst,rasterio.open(row['jrc']) as src:\n        a=src.read(1); raw_valid=src.read_masks(1)>0\n        # Dataset JRC masks use binary water labels. Fail closed on other semantics.\n        if not set(np.unique(a[raw_valid])).issubset({-1,0,1}): raise ValueError('JRC is not binary; inspect source semantics')\n        raw_valid &= np.isin(a,[0,1])\n        result=np.full(dst.shape,-1,dtype='int16')\n        reproject(np.where(raw_valid,a,-1).astype('int16'),result,src_transform=src.transform,src_crs=src.crs,src_nodata=-1,dst_transform=dst.transform,dst_crs=dst.crs,dst_nodata=-1,resampling=Resampling.nearest)\n    return result==1,result>=0\n\ndef area_km2(mask,source):\n    from pyproj import Geod,Transformer\n    with rasterio.open(source) as ds:\n        transform=ds.transform; crs=ds.crs\n    geod=Geod(ellps='WGS84'); converter=Transformer.from_crs(crs,'EPSG:4326',always_xy=True); total=0.\n    # Vectorize equal-area-within-latitude rows on north-up geographic rasters.\n    if crs.to_epsg()==4326 and transform.b==0 and transform.d==0:\n        for row,n in enumerate(mask.sum(1)):\n            if not n: continue\n            pts=[transform*(c,r) for c,r in [(0,row),(1,row),(1,row+1),(0,row+1)]]\n            area,_=geod.polygon_area_perimeter(*zip(*pts)); total+=abs(area)*int(n)\n    else:\n        for r,c in zip(*np.where(mask)):\n            pts=[converter.transform(*(transform*(cc,rr))) for cc,rr in [(c,r),(c+1,r),(c+1,r+1),(c,r+1)]]\n            area,_=geod.polygon_area_perimeter(*zip(*pts)); total+=abs(area)\n    return total/1e6\n\ndef panel(row,prob,threshold,path):\n    import matplotlib.pyplot as plt\n    from matplotlib.colors import ListedColormap\n    x,y,v=read_pair(row); pred=prob>=threshold\n    errors=np.zeros(v.shape,dtype=int); errors[pred&(y==1)&v]=1; errors[pred&(y==0)&v]=2; errors[~pred&(y==1)&v]=3; errors[~v]=4\n    fig,axes=plt.subplots(2,3,figsize=(13,9))\n    views=[(x[0],'VV (dB)','gray'),(x[1],'VH (dB)','gray'),(np.where(v,y,np.nan),'Water ground truth','Blues'),(np.where(v,prob,np.nan),'Water probability','viridis'),(np.where(v,pred,np.nan),'Predicted water','Blues'),(errors,'TN black / TP green / FP red / FN blue / invalid gray',ListedColormap(['black','limegreen','red','royalblue','gray']))]\n    for ax,(a,title,cmap) in zip(axes.flat,views):\n        ax.imshow(a,cmap=cmap,vmin=0 if a is errors else None,vmax=4 if a is errors else None); ax.set_title(title,fontsize=9); ax.axis('off')\n    fig.suptitle(row['chip_id']); fig.tight_layout(); fig.savefig(path,dpi=200); plt.close(fig)\n\ndef evaluate(df,out,checkpoint=None):\n    out=Path(out); out.mkdir(parents=True,exist_ok=True)\n    if checkpoint:\n        model,norm,device=load_model(checkpoint); threshold=choose_threshold(df,model,norm,device,out)\n    else: model=None; threshold=.5\n    records=[]\n    for row in df[df.split.isin(['test','bolivia'])].to_dict('records'):\n        x,y,v=read_pair(row)\n        _,image_valid=read_image(row)\n        prob=predict(model,x,image_valid,norm,device) if model is not None else otsu(x,image_valid).astype('float32')\n        c=counts(prob>=threshold,y,v)\n        records.append({'chip_id':row['chip_id'],'event':row['event'],'split':row['split'],**metrics(c)})\n        export_raster(row['image'],out/'probabilities'/(row['chip_id']+'.tif'),prob,image_valid,True)\n    table=pd.DataFrame(records); table.to_csv(out/'per_tile.csv',index=False)\n    summary={}\n    for split,g in table.groupby('split'):\n        summary[split]=metrics(g[['tp','fp','fn','tn']].sum().to_numpy())\n        summary[split]['defined_tile_iou']=int(g.iou.notna().sum())\n    write_json(out/'metrics.json',summary)\n    events=[]\n    for (split,event),g in table.groupby(['split','event']): events.append({'split':split,'event':event,**metrics(g[['tp','fp','fn','tn']].sum().to_numpy())})\n    pd.DataFrame(events).to_csv(out/'per_event.csv',index=False)\n    selection=[]\n    for split,g in table.groupby('split'):\n        good=g.dropna(subset=['iou']).sort_values('iou')\n        if len(good): selection.extend([good.iloc[0].chip_id,good.iloc[len(good)//2].chip_id,good.iloc[-1].chip_id])\n        selection.extend(g.sample(min(2,len(g)),random_state=42).chip_id)\n    for chip in dict.fromkeys(selection):\n        row=df[df.chip_id==chip].iloc[0].to_dict()\n        with rasterio.open(out/'probabilities'/(chip+'.tif')) as src: prob=src.read(1)\n        panel(row,prob,threshold,out/(chip+'_comparison.png'))\n    write_json(out/'figure_selection.json',{'method':'worst, median, best IoU and 2 fixed-random tiles per split','chip_ids':list(dict.fromkeys(selection))})\n    return summary\n\ndef flood_maps(df,evaluation_dir,out,threshold):\n    evaluation_dir=Path(evaluation_dir); out=Path(out); out.mkdir(parents=True,exist_ok=True); records=[]\n    for row in df[df.split.isin(['test','bolivia'])].to_dict('records'):\n        path=evaluation_dir/'probabilities'/(row['chip_id']+'.tif')\n        with rasterio.open(path) as src: prob=src.read(1); valid=src.read_masks(1)>0\n        permanent,jvalid=aligned_jrc(row); valid &= jvalid; flood=(prob>=threshold)&~permanent&valid\n        export_raster(row['image'],out/(row['chip_id']+'_flood_estimate.tif'),flood,valid)\n        export_raster(row['image'],out/(row['chip_id']+'_water.tif'),prob>=threshold,valid)\n        records.append({'chip_id':row['chip_id'],'split':row['split'],'estimated_flood_km2':area_km2(flood,row['image'])})\n    pd.DataFrame(records).to_csv(out/'estimated_areas.csv',index=False)\n    write_json(out/'limitations.json',{'interpretation':'Water outside JRC permanent water; not independent confirmation of flood. Seasonal water and JRC errors remain.','independent_flood_accuracy':None})\n    return pd.DataFrame(records)\n\ndef learning_curves(run_dir):\n    import matplotlib.pyplot as plt\n    run_dir=Path(run_dir); h=pd.read_csv(run_dir/'history.csv'); fig,axes=plt.subplots(1,2,figsize=(10,4))\n    axes[0].plot(h.epoch,h.train_loss,label='train'); axes[0].plot(h.epoch,h.valid_loss,label='valid'); axes[0].set_title('BCE + Dice loss'); axes[0].legend()\n    axes[1].plot(h.epoch,h.valid_iou,label='IoU'); axes[1].plot(h.epoch,h.valid_f1,label='F1'); axes[1].legend(); axes[1].set_title('Validation water segmentation')\n    for ax in axes: ax.set_xlabel('Epoch'); ax.grid(alpha=.2)\n    fig.tight_layout(); fig.savefig(run_dir/'learning_curves.png',dpi=300); plt.close(fig)\n", 'flood/models.py': "import torch\nfrom torch import nn\nimport torch.nn.functional as F\n\ndef block(a,b):\n    return nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True),nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True))\n\nclass PlainUNet(nn.Module):\n    def __init__(self):\n        super().__init__(); widths=[32,64,128,256,512]\n        self.enc=nn.ModuleList([block(a,b) for a,b in zip([2]+widths[:-1],widths)])\n        self.dec=nn.ModuleList([block(a+b,b) for a,b in zip(widths[:0:-1],widths[-2::-1])])\n        self.head=nn.Conv2d(32,1,1)\n    def forward(self,x):\n        skips=[]\n        for i,layer in enumerate(self.enc):\n            if i: x=F.max_pool2d(x,2)\n            x=layer(x); skips.append(x)\n        for layer,skip in zip(self.dec,skips[-2::-1]):\n            x=layer(torch.cat([F.interpolate(x,size=skip.shape[-2:],mode='bilinear',align_corners=False),skip],1))\n        return self.head(x)\n\ndef build_model(name='resnet34',pretrained=True):\n    if name=='plain': return PlainUNet()\n    if name!='resnet34': raise ValueError(name)\n    import segmentation_models_pytorch as smp\n    return smp.Unet(encoder_name='resnet34',encoder_weights='imagenet' if pretrained else None,in_channels=2,classes=1,activation=None)\n", 'flood/train.py': 'import json, random, time, platform, subprocess\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader, Subset\nfrom .models import build_model\nfrom .data import FloodDataset\nfrom .core import masked_loss, counts, metrics, write_json\n\nDEFAULT=dict(model=\'resnet34\',pretrained=True,seed=42,batch_size=8,workers=2,epochs=100,patience=15,lr=1e-4,weight_decay=1e-4)\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n\ndef save_checkpoint(path,state):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    temp=path.with_suffix(\'.tmp\'); torch.save(state,temp); temp.replace(path)\n\ndef environment(out):\n    out=Path(out); out.mkdir(parents=True,exist_ok=True)\n    import sys\n    versions=subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True)\n    (out/\'requirements-lock.txt\').write_text(versions,encoding=\'utf-8\')\n    info=dict(python=platform.python_version(),torch=torch.__version__,cuda=torch.cuda.is_available(),gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)\n    write_json(out/\'environment.json\',info); return info\n\ndef epoch(model,loader,device,optimizer=None,scaler=None):\n    training=optimizer is not None; model.train(training); total=np.zeros(4,dtype=np.int64); losses=[]\n    for batch in loader:\n        x=batch[\'image\'].to(device); y=batch[\'target\'].to(device); v=batch[\'valid\'].to(device)\n        if training: optimizer.zero_grad(set_to_none=True)\n        with torch.set_grad_enabled(training),torch.autocast(device_type=device.type,enabled=device.type==\'cuda\'):\n            z=model(x); loss=masked_loss(z,y,v)\n        if loss is None: continue\n        if not torch.isfinite(loss): raise FloatingPointError(\'Non-finite loss\')\n        if training:\n            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()\n        losses.append(float(loss.detach())); total+=counts(z.detach().sigmoid().cpu().numpy()>=.5,y.cpu().numpy(),v.cpu().numpy())\n    if not losses: raise ValueError(\'Empty valid dataset\')\n    return {\'loss\':float(np.mean(losses)),**metrics(total)}\n\ndef overfit_check(df,norm,out,steps=80,device=None):\n    device=torch.device(device or (\'cuda\' if torch.cuda.is_available() else \'cpu\'))\n    seed_all(42); dataset=FloodDataset(df,\'train\',norm,False)\n    loader=DataLoader(Subset(dataset,list(range(min(4,len(dataset))))),batch_size=2,shuffle=False)\n    model=build_model(\'resnet34\',False).to(device); optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\'); history=[]; start=time.time()\n    for i in range(steps):\n        m=epoch(model,loader,device,optimizer,scaler); history.append(m[\'loss\'])\n    result={\'first_loss\':history[0],\'last_loss\':history[-1],\'seconds\':time.time()-start,\'steps\':steps}\n    write_json(Path(out)/\'overfit_check.json\',result)\n    if history[-1]>=history[0]*.7: raise RuntimeError(\'Tiny-set overfit check failed; inspect data and model\')\n    del model,optimizer\n    if device.type==\'cuda\': torch.cuda.empty_cache()\n    return result\n\ndef train(df,norm,out,config=None,resume=True,allow_cpu=False):\n    cfg={**DEFAULT,**(config or {})}; out=Path(out); out.mkdir(parents=True,exist_ok=True)\n    if not torch.cuda.is_available() and not allow_cpu: raise RuntimeError(\'Select GPU in Colab; full CPU training is disabled\')\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\'); seed_all(cfg[\'seed\'])\n    trainset=FloodDataset(df,\'train\',norm,True); valset=FloodDataset(df,\'valid\',norm,False)\n    generator=torch.Generator().manual_seed(cfg[\'seed\'])\n    loader=DataLoader(trainset,batch_size=cfg[\'batch_size\'],shuffle=True,num_workers=cfg[\'workers\'],generator=generator,pin_memory=device.type==\'cuda\')\n    valid=DataLoader(valset,batch_size=cfg[\'batch_size\'],num_workers=cfg[\'workers\'])\n    last=out/\'last.pt\'; state=None\n    if resume and last.exists():\n        state=torch.load(last,map_location=\'cpu\',weights_only=False) # Only our own checkpoint.\n        for key in cfg:\n            if key not in [\'epochs\',\'workers\'] and state[\'config\'][key]!=cfg[key]: raise ValueError(\'Resume config mismatch: \'+key)\n        if state[\'normalization\']!=norm: raise ValueError(\'Resume normalization mismatch\')\n        old_ids=state[\'split_ids\']; ids={s:sorted(df[df.split==s].chip_id.tolist()) for s in df.split.unique()}\n        if old_ids!=ids: raise ValueError(\'Resume dataset mismatch\')\n    model=build_model(cfg[\'model\'],cfg[\'pretrained\'] and state is None).to(device)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg[\'lr\'],weight_decay=cfg[\'weight_decay\'])\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\')\n    history=[]; best=-1.; stale=0; start_epoch=0\n    if state:\n        model.load_state_dict(state[\'model\']); optimizer.load_state_dict(state[\'optimizer\']); scaler.load_state_dict(state[\'scaler\'])\n        history=state[\'history\']; best=state[\'best\']; stale=state[\'stale\']; start_epoch=state[\'epoch\']+1\n        random.setstate(state[\'python_rng\']); np.random.set_state(state[\'numpy_rng\']); torch.set_rng_state(state[\'torch_rng\'])\n        generator.set_state(state[\'loader_rng\'])\n        if device.type==\'cuda\' and state[\'cuda_rng\']: torch.cuda.set_rng_state_all(state[\'cuda_rng\'])\n    environment(out); write_json(out/\'config.json\',cfg); write_json(out/\'normalization.json\',norm)\n    print(f\'{cfg["model"]}: {len(trainset)} train patches; device={device}; resume epoch={start_epoch}\',flush=True)\n    for e in range(start_epoch,cfg[\'epochs\']):\n        if stale>=cfg[\'patience\']: break\n        t=time.time(); tr=epoch(model,loader,device,optimizer,scaler); va=epoch(model,valid,device)\n        if not np.isfinite(va[\'iou\']): raise ValueError(\'Validation IoU undefined\')\n        improved=va[\'iou\']>best\n        if improved: best=va[\'iou\']; stale=0\n        else: stale+=1\n        record={\'epoch\':e+1,\'seconds\':time.time()-t,**{\'train_\'+k:v for k,v in tr.items()},**{\'valid_\'+k:v for k,v in va.items()}}\n        history.append(record)\n        checkpoint=dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),epoch=e,best=best,stale=stale,history=history,config=cfg,normalization=norm,python_rng=random.getstate(),numpy_rng=np.random.get_state(),torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all() if device.type==\'cuda\' else [],loader_rng=generator.get_state(),split_ids={s:sorted(df[df.split==s].chip_id.tolist()) for s in df.split.unique()})\n        if improved: save_checkpoint(out/\'best.pt\',checkpoint)\n        save_checkpoint(last,checkpoint); pd.DataFrame(history).to_csv(out/\'history.csv\',index=False)\n        print(f\'Epoch {e+1}: train loss={tr["loss"]:.4f}, valid IoU={va["iou"]:.4f}, {record["seconds"]:.1f}s\',flush=True)\n    return out/\'best.pt\'\n', 'flood/__init__.py': '"""Reproducible Sen1Floods11 water segmentation."""\n', 'requirements-colab.txt': '# Keep the Colab-provided compatible torch/torchvision pair.\nsegmentation-models-pytorch==0.5.0\nrasterio>=1.4,<2\nscikit-image>=0.24,<1\npandas>=2,<3\nmatplotlib>=3.8,<4\nrequests>=2.31,<3\npyproj>=3.6,<4\n', 'tests/test_core.py': "import unittest\nimport numpy as np\nimport torch\nfrom flood.core import counts, metrics, masked_loss, assemble, validate_splits\n\nclass CoreTests(unittest.TestCase):\n    def test_metrics_and_invalid(self):\n        y = np.array([1, 0, 1, 0]); p = np.array([1, 1, 0, 1]); v = np.array([1, 1, 1, 0], bool)\n        c = counts(p, y, v)\n        self.assertEqual(c.tolist(), [1, 1, 1, 0])\n        self.assertAlmostEqual(metrics(c)['iou'], 1/3)\n        self.assertAlmostEqual(metrics(c)['f1'], .5)\n\n    def test_loss_ignores_invalid(self):\n        y = torch.tensor([[[[1., 0.]]]])\n        v = torch.tensor([[[[True, False]]]])\n        a = torch.tensor([[[[0., -100.]]]], requires_grad=True)\n        b = torch.tensor([[[[0., 100.]]]])\n        self.assertEqual(masked_loss(a,y,v).item(), masked_loss(b,y,v).item())\n        masked_loss(a,y,v).backward()\n        self.assertEqual(a.grad[0,0,0,1].item(),0)\n        self.assertIsNone(masked_loss(a,y,torch.zeros_like(v)))\n\n    def test_empty_target_penalized(self):\n        y = torch.zeros(1,1,4,4); v = torch.ones_like(y,dtype=torch.bool)\n        self.assertGreater(masked_loss(y+5,y,v),masked_loss(y-5,y,v))\n\n    def test_assemble(self):\n        tiles = [np.full((2,2),i) for i in range(4)]\n        out = assemble(tiles,4,2)\n        np.testing.assert_array_equal(out,[[0,0,1,1],[0,0,1,1],[2,2,3,3],[2,2,3,3]])\n\n    def test_split_leakage(self):\n        with self.assertRaises(ValueError):\n            validate_splits([{'chip_id':'a','split':'train','event':'Spain'}, {'chip_id':'a','split':'test','event':'Spain'}])\n        with self.assertRaises(ValueError):\n            validate_splits([{'chip_id':'a','split':'train','event':'Bolivia'}])\n\nif __name__ == '__main__': unittest.main()\n", 'tests/test_pipeline.py': "import tempfile, unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\nimport numpy as np\nimport pandas as pd\nimport rasterio\nfrom rasterio.transform import from_origin\nimport torch\nfrom flood.data import audit,FloodDataset,read_image\nfrom flood.evaluate import export_raster,aligned_jrc,area_km2\nfrom flood.models import build_model\nfrom flood.train import train\n\nclass PipelineTests(unittest.TestCase):\n    def setUp(self):\n        torch.set_num_threads(2)\n        self.temp=tempfile.TemporaryDirectory(); self.root=Path(self.temp.name); rows=[]\n        for i,split in enumerate(['train','valid']):\n            row=dict(chip_id=f'Spain_{i}',event='Spain',split=split)\n            for key in ['image','label','jrc']:\n                row[key]=str(self.root/f'{i}_{key}.tif')\n                arr=np.ones((2 if key=='image' else 1,512,512),dtype='float32')\n                if key=='image': arr*=(-10 if split=='train' else 100)\n                if key=='label': arr[:,:,256:]=0; arr[:,:256,:256]=-1\n                with rasterio.open(row[key],'w',driver='GTiff',height=512,width=512,count=len(arr),dtype='float32',crs='EPSG:4326',transform=from_origin(10,10,.0001,.0001)) as dst: dst.write(arr)\n            rows.append(row)\n        # Move validation spatially so synthetic split geometry is independent.\n        for key in ['image','label','jrc']:\n            with rasterio.open(rows[1][key],'r+') as dst: dst.transform=from_origin(11,10,.0001,.0001)\n        self.df=pd.DataFrame(rows)\n        self.norm={'mean':[-10.,-10.],'std':[1.,1.]}\n    def tearDown(self): self.temp.cleanup()\n\n    def test_normalization_train_only_and_invalid_patch(self):\n        norm,_=audit(self.df,self.root/'audit')\n        self.assertEqual(norm['mean'],[-10.,-10.])\n        ds=FloodDataset(self.df,'train',self.norm)\n        self.assertEqual(len(ds),3); self.assertEqual(ds.skipped,1)\n        self.assertEqual(ds[0]['image'].shape,(2,256,256))\n\n    def test_joint_augmentation_alignment(self):\n        row=self.df.iloc[0]\n        with rasterio.open(row.label) as ds: mask=ds.read(1)\n        mask[256:384,:128]=0\n        with rasterio.open(row.label,'r+') as ds: ds.write(mask,1)\n        with rasterio.open(row.image,'r+') as ds: ds.write(np.stack([mask,mask]).astype('float32'))\n        ds=FloodDataset(self.df,'train',{'mean':[0,0],'std':[1,1]},augment=True)\n        for _ in range(8):\n            b=ds[1]; v=b['valid'][0]\n            torch.testing.assert_close(b['image'][0][v],b['target'][0][v])\n\n    def test_annotation_nodata_does_not_change_input(self):\n        before=FloodDataset(self.df,'train',self.norm)[1]['image'].clone()\n        row=self.df.iloc[0]\n        with rasterio.open(row.label,'r+') as ds:\n            a=ds.read(1); a[300:320,40:60]=-1; ds.write(a,1)\n        after=FloodDataset(self.df,'train',self.norm)[1]['image']\n        torch.testing.assert_close(before,after)\n        _,image_valid=read_image(row)\n        self.assertTrue(image_valid.all())\n\n    def test_export_and_jrc_alignment(self):\n        row=self.df.iloc[0].to_dict(); v=np.ones((512,512),bool); v[0,0]=False\n        p=self.root/'output.tif'; export_raster(row['image'],p,np.ones((512,512)),v)\n        with rasterio.open(p) as ds:\n            self.assertEqual(ds.nodata,255); self.assertEqual(ds.read(1)[0,0],255); self.assertEqual(ds.crs.to_epsg(),4326)\n        # A coarse JRC grid covering the same extent must align correctly.\n        with rasterio.open(row['jrc'],'w',driver='GTiff',height=256,width=256,count=1,dtype='int16',crs='EPSG:4326',transform=from_origin(10,10,.0002,.0002)) as ds: ds.write(np.ones((256,256),dtype='int16'),1)\n        permanent,valid=aligned_jrc(row)\n        self.assertTrue(permanent.all()); self.assertTrue(valid.all())\n        self.assertGreater(area_km2(v,row['image']),0)\n\n    def test_models_forward_backward(self):\n        for name in ['plain','resnet34']:\n            model=build_model(name,False).eval(); x=torch.randn(1,2,32,32)\n            z=model(x); self.assertEqual(z.shape,(1,1,32,32)); z.mean().backward()\n            del model\n\n    def test_resume_continues_epoch(self):\n        cfg={'model':'plain','pretrained':False,'epochs':1,'workers':0,'batch_size':2}\n        out=self.root/'run'\n        with patch('flood.train.build_model',side_effect=lambda *a:torch.nn.Conv2d(2,1,1)),patch('flood.train.environment',return_value={}):\n            train(self.df,self.norm,out,cfg,allow_cpu=True)\n            train(self.df,self.norm,out,{**cfg,'epochs':2},allow_cpu=True)\n        state=torch.load(out/'last.pt',weights_only=False)\n        self.assertEqual(state['epoch'],1); self.assertEqual(len(state['history']),2)\n\nif __name__=='__main__': unittest.main()\n"}
for name, source in SOURCES.items():
    dest = WORK / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(source, encoding='utf-8')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])
import torch
from flood.train import environment
print(environment(WORK / 'environment'))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — audit only; select GPU before training')


## 2. Постоянное хранилище

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE = Path('/content/drive/MyDrive/FloodSegmentation')
SAVE.mkdir(parents=True, exist_ok=True)
(SAVE / 'write_check.txt').write_text('Storage available', encoding='utf-8')
assert (SAVE / 'write_check.txt').read_text(encoding='utf-8') == 'Storage available'
environment(SAVE / 'environment')
print('Persistent results:', SAVE)


## 3. Проверки реализации

In [ ]:
subprocess.check_call([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'])


## 4. Загрузка, аудит, train-only нормализация

In [ ]:
from flood.data import prepare, audit, eda
DATA = WORK / 'data'
df = prepare(DATA, include_jrc=True)
assert len(df) == 446, f'Unexpected manifest size: {len(df)} — inspect before training'
norm, stats = audit(df, SAVE / 'data_audit')
df.to_csv(SAVE / 'data_audit' / 'manifest.csv', index=False)
eda(df, SAVE / 'data_audit')
display(df.groupby('split').size().rename('tiles').to_frame())
display(stats.groupby(['split', 'event'])[['valid_pixels','water_pixels']].sum())
print('Train-only normalization:', norm)


In [ ]:
from IPython.display import display, Image
for p in sorted((SAVE / 'data_audit').glob('*_data.png'))[:4]:
    display(Image(filename=str(p)))


## 5. Проверка переобучения на нескольких патчах

In [ ]:
assert torch.cuda.is_available(), 'Runtime → Change runtime type → GPU'
from flood.train import overfit_check
check_path = SAVE / 'checks'
check_path.mkdir(exist_ok=True)
print(overfit_check(df, norm, check_path, steps=80))


## 6. Основное обучение
До 100 эпох; early stopping patience=15. Не меняйте конфигурацию существующего запуска при resume.

In [ ]:
from flood.train import train, DEFAULT
CONFIG = {**DEFAULT, 'batch_size': 8, 'workers': 2, 'epochs': 100}
# Reduce batch_size to 4 or 2 if the first run runs out of GPU memory.
# Keep a separate run name when changing the experiment configuration.
RUN = SAVE / 'runs' / 'resnet34_seed42'
checkpoint = train(df, norm, RUN, CONFIG, resume=True)
print('Best checkpoint:', checkpoint)


## 7. Независимое тестирование
Порог выбирается только на validation. Bolivia — отдельный holdout.

In [ ]:
from flood.evaluate import evaluate, learning_curves
RUN = SAVE / 'runs' / 'resnet34_seed42'
checkpoint = RUN / 'best.pt'
assert checkpoint.exists(), 'Train the main model first'
main_metrics = evaluate(df, SAVE / 'evaluation' / 'resnet34', checkpoint)
learning_curves(RUN)
print(json.dumps(main_metrics, indent=2))
display(Image(filename=str(RUN / 'learning_curves.png')))


## 8. Пороговый baseline Otsu

In [ ]:
from flood.evaluate import evaluate
otsu_metrics = evaluate(df, SAVE / 'evaluation' / 'otsu')
print(json.dumps(otsu_metrics, indent=2))


## 9. Обычная U-Net
Отдельный полноценный эксперимент; может потребовать следующей GPU-сессии.

In [ ]:
from flood.train import train, DEFAULT
from flood.evaluate import evaluate, learning_curves
plain_run = SAVE / 'runs' / 'plain_seed42'
plain_checkpoint = train(df, norm, plain_run, {**DEFAULT, 'model':'plain', 'pretrained':False}, resume=True)
plain_metrics = evaluate(df, SAVE / 'evaluation' / 'plain', plain_checkpoint)
learning_curves(plain_run)
print(json.dumps(plain_metrics, indent=2))


## 10. Карты предполагаемого затопления
JRC применяется только после сегментации. Возможны сезонная вода и ошибки карты постоянных водоёмов.

In [ ]:
from flood.evaluate import flood_maps
eval_dir = SAVE / 'evaluation' / 'resnet34'
threshold = json.loads((eval_dir / 'threshold.json').read_text())['threshold']
areas = flood_maps(df, eval_dir, SAVE / 'maps', threshold)
display(areas.head())
print('These are estimates of water outside JRC permanent water, not independently verified flood labels.')


## 11. Итоговая таблица и рисунки

In [ ]:
import pandas as pd
rows = []
for name in ['otsu','plain','resnet34']:
    p = SAVE / 'evaluation' / name / 'metrics.json'
    if not p.exists():
        print('Not completed:', name)
        continue
    for split, values in json.loads(p.read_text()).items():
        rows.append({'model':name, 'split':split, **values})
comparison = pd.DataFrame(rows)
comparison.to_csv(SAVE / 'comparison.csv', index=False)
display(comparison)
for p in sorted((SAVE / 'evaluation' / 'resnet34').glob('*_comparison.png'))[:5]:
    display(Image(filename=str(p)))
print('Results are saved in', SAVE)
